In [ ]:
import sys

#FIXME: point to local copy of dinov3 repo
REPO_DIR = "/home/teun/Projects/maritime-object-detection/2026Q2_innovation_sprint/dinov3"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from contextlib import nullcontext
from PIL import Image
import torch
from torchvision.transforms import v2
import matplotlib.pyplot as plt
from matplotlib import colormaps
from functools import partial
from dinov3.eval.segmentation.inference import make_inference

# Setup
Loads large DinoV3 model (vit7b16)

In [ ]:

def get_img():
    import requests
    url = "http://images.cocodataset.org/val2017/000000039769.jpg"
    image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
    return image

def make_transform(resize_size: int | list[int] = 768):
    to_tensor = v2.ToImage()
    resize = v2.Resize((resize_size, resize_size), antialias=True)
    to_float = v2.ToDtype(torch.float32, scale=True)
    normalize = v2.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )
    return v2.Compose([to_tensor, resize, to_float, normalize])

segmentor = torch.hub.load(
    REPO_DIR, 'dinov3_vit7b16_ms', source="local", 
    weights="https://dl.fbaipublicfiles.com/dinov3/dinov3_vit7b16/dinov3_vit7b16_ade20k_m2f_head-bf307cb1.pth", 
    backbone_weights="https://dl.fbaipublicfiles.com/dinov3/dinov3_vitb16/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"
    )

# If vit7b16 is too large, use EOMT segmentor instead

In [ ]:
import requests
import torch
from PIL import Image

from transformers import AutoImageProcessor, AutoModelForUniversalSegmentation

model_id = "tue-mps/eomt-dinov3-coco-panoptic-base-640"
processor = AutoImageProcessor.from_pretrained(model_id)
model = AutoModelForUniversalSegmentation.from_pretrained(model_id).to("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
image = Image.open(requests.get(
    # "https://www.defensenews.com/resizer/eAkKGXbfU5W7oP-uJdvChar4POw=/1024x0/filters:format(jpg):quality(70)/cloudfront-us-east-1.images.arcpublishing.com/archetype/MUGTFAQE7FDGLMZUTCEIV6TSPY.jpg", 
    "https://i.redd.it/exk38l3cekj91.jpg",
    stream=True
    ).raw)


inputs = processor(images=image, return_tensors="pt").to(model.device)

with torch.inference_mode():
    outputs = model(**inputs)

segmentation = processor.post_process_panoptic_segmentation(outputs, target_sizes=[image.size[::-1]])[0]

In [ ]:
# Build and display a colored overlay from the HF post-processed panoptic output
import numpy as np
from PIL import Image as PILImage
import matplotlib.pyplot as plt

# `segmentation` is produced by the previous cell:
# segmentation = processor.post_process_panoptic_segmentation(...)[0]
seg_map = segmentation["segmentation"]

# Convert to numpy array if it's a PIL Image
if isinstance(seg_map, PILImage.Image):
    seg_arr = np.array(seg_map)
else:
    seg_arr = np.array(seg_map)

# If seg_arr is a 2D label map, map labels to colors; otherwise assume it's an RGB image
if seg_arr.ndim == 2:
    labels = seg_arr
    unique = np.unique(labels)
    cmap = plt.get_cmap("tab20")
    # assign deterministic colors for each label
    color_map = {lab: cmap(i % cmap.N)[:3] for i, lab in enumerate(unique)}
    colored = np.zeros((labels.shape[0], labels.shape[1], 3), dtype=np.float32)
    for lab in unique:
        mask = labels == lab
        colored[mask] = color_map[lab]
else:
    # RGB or RGBA image
    if seg_arr.shape[2] == 4:
        seg_arr = seg_arr[..., :3]
    colored = seg_arr.astype(np.float32) / 255.0

# Original image (from previous cell) is `image` (PIL Image)
img_arr = np.array(image).astype(np.float32) / 255.0
# Resize original image if sizes differ
if img_arr.shape[0] != colored.shape[0] or img_arr.shape[1] != colored.shape[1]:
    img_arr = np.array(PILImage.fromarray((img_arr * 255).astype(np.uint8)).resize((colored.shape[1], colored.shape[0]))) / 255.0

# Blend and display
alpha = 0.5
overlay = np.clip(alpha * colored + (1 - alpha) * img_arr, 0, 1)

plt.figure(figsize=(12, 6))
plt.subplot(121)
plt.imshow(image)
plt.axis('off')
plt.title('Original')
plt.subplot(122)
plt.imshow(overlay)
plt.axis('off')
plt.title('Segmentation overlay')

In [ ]:
import numpy as np
from PIL import Image as PILImage
from pathlib import Path
import matplotlib.pyplot as plt

# Params
margin = 1.25            # expand bbox by this factor
min_side = 64            # skip tiny crops
overlap_thresh = 0.25    # keep refined segment if overlap with original instance >= this fraction
out_dir = Path("outputs/refined")
out_dir.mkdir(parents=True, exist_ok=True)

# Prepare original segmentation
seg_map = np.array(segmentation["segmentation"])          # 2D int map of segment ids
segments_info = segmentation.get("segments_info", segmentation.get("segments", []))

H, W = seg_map.shape
refined_canvas = np.zeros((H, W), dtype=np.int32)        # store refined instance ids (choose new ids)
next_refined_id = 1

def bbox_from_mask(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    x0, x1 = int(xs.min()), int(xs.max())
    y0, y1 = int(ys.min()), int(ys.max())
    return x0, y0, x1, y1

for seg in segments_info:
    sid = seg.get("id", seg.get("segment_id"))
    mask = (seg_map == sid)
    bbox = bbox_from_mask(mask)
    if bbox is None:
        continue
    x0, y0, x1, y1 = bbox
    w = x1 - x0 + 1
    h = y1 - y0 + 1
    # expand bbox
    cx, cy = x0 + w/2, y0 + h/2
    new_w = max(min_side, int(w * margin))
    new_h = max(min_side, int(h * margin))
    nx0 = max(0, int(cx - new_w//2))
    ny0 = max(0, int(cy - new_h//2))
    nx1 = min(W, nx0 + new_w)
    ny1 = min(H, ny0 + new_h)
    crop = image.crop((nx0, ny0, nx1, ny1))
    crop_size = (ny1 - ny0, nx1 - nx0)  # (height, width)

    # Run processor+model on crop
    inputs = processor(images=crop, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        outputs = model(**inputs)
    refined = processor.post_process_panoptic_segmentation(outputs, target_sizes=[crop_size])[0]
    refined_seg = np.array(refined["segmentation"])

    # For each refined segment in crop, test overlap with original instance mask (cropped)
    refined_info = refined.get("segments_info", refined.get("segments", []))
    orig_mask_crop = np.array(mask[ny0:ny1, nx0:nx1])  # boolean

    for r in refined_info:
        rid = r.get("id", r.get("segment_id"))
        rmask = (refined_seg == rid)
        if rmask.sum() == 0:
            continue
        # compute overlap with original instance mask (in crop coords)
        inter = (rmask & orig_mask_crop).sum()
        frac = inter / float(orig_mask_crop.sum() + 1e-9)
        if frac >= overlap_thresh:
            # accept this refined segment: paste into canvas with new id
            paste_mask = rmask
            refined_canvas[ny0:ny1, nx0:nx1][paste_mask] = next_refined_id
            # optional: save crop overlay for inspection
            cmap = plt.get_cmap("tab20")
            color = np.array(cmap(next_refined_id % cmap.N)[:3]) * 255
            overlay = (0.5 * (np.array(crop).astype(np.float32)) + 0.5 * (color.reshape(1,1,3) * paste_mask[...,None])).astype(np.uint8)
            PILImage.fromarray(overlay).save(out_dir / f"crop_refined_{sid}_{next_refined_id}.png")
            next_refined_id += 1

# Visualize aggregated refined masks overlayed on original image
color_palette = plt.get_cmap("tab20")
H, W = refined_canvas.shape
color_img = np.zeros((H, W, 3), dtype=np.float32)
unique_ids = np.unique(refined_canvas)
for i, rid in enumerate(unique_ids):
    if rid == 0:
        continue
    mask = refined_canvas == rid
    color = np.array(color_palette(i % color_palette.N)[:3])
    color_img[mask] = color

img_arr = np.array(image).astype(np.float32) / 255.0
blended = 0.5 * color_img + 0.5 * img_arr
blended = (np.clip(blended, 0, 1) * 255).astype(np.uint8)

plt.figure(figsize=(12,6))
plt.subplot(121)
plt.imshow(image)
plt.axis('off')
plt.title('Original')
plt.subplot(122)
plt.imshow(blended)
plt.axis('off')
plt.title('Refined instances overlay')
PILImage.fromarray(blended).save(out_dir / "refined_aggregated_overlay.png")
print('Saved refined crops and aggregated overlay to', out_dir)

# Same setup using EOMT Large

In [ ]:
import matplotlib.pyplot as plt
import requests
import torch
from PIL import Image

from transformers import EomtForUniversalSegmentation, AutoImageProcessor


model_id = "tue-mps/coco_panoptic_eomt_large_640"
processor = AutoImageProcessor.from_pretrained(model_id)
model = EomtForUniversalSegmentation.from_pretrained(model_id)

In [ ]:
image = Image.open("/home/teun/Projects/maritime-object-detection/2026Q2_innovation_sprint/47b816b2-b314-4489-ba75-06ecfbeb71c8.jpeg")

inputs = processor(
    images=image,
    return_tensors="pt",
)

with torch.inference_mode():
    outputs = model(**inputs)

# Prepare the original image size in the format (height, width)
target_sizes = [(image.height, image.width)]

# Post-process the model outputs to get final segmentation prediction
preds = processor.post_process_panoptic_segmentation(
    outputs,
    target_sizes=target_sizes,
)

# Visualize the panoptic segmentation mask
plt.figure(figsize=(12,6))
plt.subplot(121)
plt.imshow(image)
plt.axis('off')
plt.title('Original')
plt.subplot(122)
plt.imshow(preds[0]["segmentation"])
plt.axis("off")
plt.title("Panoptic Segmentation")
plt.show()
